# Lab 5 — Speech Enhancement, Speaker Recognition và Speaker Separation

Notebook thực hiện ba phần: khử nhiễu trong miền STFT, nhận dạng/xác minh người nói bằng MFCC, và tách hai nguồn bằng FastICA. Mỗi task nằm trong một nhóm cell riêng; chạy tuần tự từ đầu để tạo lại kết quả trong `results/`.

**Ghi chú dữ liệu:** dataset hiện có enrollment/test cho spk02 và spk03; spk01 chỉ có test nên không dùng trong speaker-ID. Một enrollment utterance mỗi speaker được giữ riêng làm development cho verification.

## Task 0 — Cấu hình và kiểm tra dữ liệu

In [29]:
from pathlib import Path
import json, warnings
import numpy as np, pandas as pd, soundfile as sf
from scipy.fft import dct
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import FastICA
from scipy.optimize import linear_sum_assignment

ROOT = Path.cwd()
LAB = ROOT if (ROOT / "lab5_dataset").exists() else ROOT / "Lab05_2351260681_NguyenMinhPhuc"
DATA = LAB / "lab5_dataset"
OUT = LAB / "results"
AUDIO, FIG = OUT / "audio", OUT / "figures"
AUDIO.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)
SR=16000; N_FFT=512; WIN=400; HOP=160
TARGET_SNRS=[0,5,10]; BETAS=[1.0,1.5,2.0]; FLOORS=[0.0,0.01,0.05]
RNG_SEED=457
CLEAN_DIR=DATA/"clean"
SPEAKERS=sorted({p.name.split("_enroll_")[0] for p in CLEAN_DIR.glob("*_enroll_*.wav") if list(CLEAN_DIR.glob(p.name.split("_enroll_")[0]+"_test_*.wav"))})
assert SPEAKERS, "No speakers with both enrollment and test files were found."
CASE_SPEAKER=SPEAKERS[0]
DEV_FILES={spk:sorted(CLEAN_DIR.glob(f"{spk}_enroll_*.wav"))[-1] for spk in SPEAKERS}
print(f"Sample rate={SR}; enrolled speakers={SPEAKERS}; dev holdout={ {k:v.name for k,v in DEV_FILES.items()} }")
print("Dataset has no manifest; speaker splits are discovered from filenames.")


Sample rate=16000; enrolled speakers=['spk01', 'spk02', 'spk03']; dev holdout={'spk01': 'spk01_enroll_04.wav', 'spk02': 'spk02_enroll_04.wav', 'spk03': 'spk03_enroll_04.wav'}
Dataset has no manifest; speaker splits are discovered from filenames.


## Task 1 — Chuẩn hóa tín hiệu, tạo nhiễu và kiểm tra SNR

In [30]:
def stft(x):
    x=np.asarray(x,dtype=float); pad=N_FFT//2
    x=np.pad(x,(pad,pad))
    frames=np.lib.stride_tricks.sliding_window_view(x,N_FFT)[::HOP]
    win=np.hanning(WIN); window=np.zeros(N_FFT); start=(N_FFT-WIN)//2; window[start:start+WIN]=win
    return np.fft.rfft(frames*window[None,:],axis=1).T

def istft(S,length):
    frames=np.fft.irfft(S.T,n=N_FFT,axis=1)
    win=np.hanning(WIN); window=np.zeros(N_FFT); start=(N_FFT-WIN)//2; window[start:start+WIN]=win
    out=np.zeros((len(frames)-1)*HOP+N_FFT); norm=np.zeros_like(out)
    for i,frame in enumerate(frames):
        lo=i*HOP; out[lo:lo+N_FFT]+=frame*window; norm[lo:lo+N_FFT]+=window**2
    nz=norm>1e-10; out[nz]/=norm[nz]
    out=out[N_FFT//2:N_FFT//2+length]
    return out

def mel_filterbank(n_mels=40):
    hz_to_mel=lambda hz:2595*np.log10(1+hz/700)
    mel_to_hz=lambda mel:700*(10**(mel/2595)-1)
    points=mel_to_hz(np.linspace(hz_to_mel(0),hz_to_mel(SR/2),n_mels+2))
    bins=np.floor((N_FFT+1)*points/SR).astype(int); fb=np.zeros((n_mels,N_FFT//2+1))
    for m in range(1,n_mels+1):
        l,c,r=bins[m-1:m+2]; c=max(c,l+1); r=max(r,c+1)
        fb[m-1,l:c]=np.arange(l,c)-l; fb[m-1,l:c]/=(c-l)
        fb[m-1,c:r]=r-np.arange(c,r); fb[m-1,c:r]/=(r-c)
    return fb
MEL_FB=mel_filterbank()
def mfcc_features(x):
    X=stft(x); power_spec=np.abs(X)**2
    mel=np.maximum(MEL_FB@power_spec,1e-10)
    return dct(10*np.log10(mel),type=2,axis=0,norm="ortho")[:20]

def delta_features(M):
    return np.gradient(M,axis=1)

def load_audio(path):
    x, sr = sf.read(path, dtype="float64")
    if x.ndim>1: x=x.mean(axis=1)
    if sr != SR: raise ValueError(f"Expected {SR} Hz, got {sr} Hz: {path}")
    return x

def power(x): return float(np.mean(np.asarray(x,dtype=np.float64)**2)+1e-15)
def rms(x): return np.sqrt(power(x))
def match_length(x,L):
    if len(x)>=L: return x[:L].copy()
    return np.tile(x, int(np.ceil(L/len(x))))[:L]
def add_noise_at_snr(s,v,snr_db):
    v=match_length(v,len(s)); alpha=np.sqrt(power(s)/(power(v)*10**(snr_db/10)))
    return s+alpha*v,alpha
def measured_snr(clean,noisy):
    L=min(len(clean),len(noisy)); e=noisy[:L]-clean[:L]
    return 10*np.log10((np.sum(clean[:L]**2)+1e-15)/(np.sum(e**2)+1e-15))
def save_audio(name,x,peak_normalize=False):
    x=np.asarray(x,dtype=np.float64)
    peak=float(np.max(np.abs(x))) if len(x) else 0.0
    if peak_normalize and peak>0:
        x=0.99*x/peak
        subtype="PCM_16"
    else:
        # Keep noisy signals and mixtures unchanged; use floating-point WAV if their peaks exceed full scale.
        subtype="FLOAT" if peak>0.999 else "PCM_16"
    sf.write(AUDIO/name,x,SR,subtype=subtype)

def plot_spec(ax,x,title):
    D=(20*np.log10(np.abs(stft(x))+1e-10) - np.max(20*np.log10(np.abs(stft(x))+1e-10)))
    im=ax.imshow(D,origin="lower",aspect="auto",extent=[0,len(x)/SR,0,SR/2],cmap="magma",vmin=-70,vmax=0)
    ax.set(title=title,xlabel="Thời gian (s)",ylabel="Tần số (Hz)")

clean=load_audio(DATA/"clean"/f"{CASE_SPEAKER}_test_01.wav")
noise=load_audio(DATA/"noise"/"fan.wav")
noisy_by_snr={}; alpha_by_snr={}
for target in TARGET_SNRS:
    y,a=add_noise_at_snr(clean,noise,target); noisy_by_snr[target]=y; alpha_by_snr[target]=a
    save_audio(f"noisy_{target}dB.wav",y)
    actual=measured_snr(clean,y)
    print(f"target={target:>2} dB | measured={actual:6.3f} dB | alpha={a:.6f} | clean RMS={rms(clean):.4f}")
    assert abs(actual-target)<=0.2
fig,axs=plt.subplots(3,2,figsize=(12,9))
for row,(title,x) in enumerate([("Clean",clean),("Noisy 0 dB",noisy_by_snr[0]),("Noisy 5 dB",noisy_by_snr[5])]):
    t=np.arange(len(x))/SR; axs[row,0].plot(t,x,lw=.5); axs[row,0].set(title=title+" waveform",xlabel="s")
    plot_spec(axs[row,1],x,title+" spectrogram")
fig.tight_layout(); fig.savefig(FIG/"clean_noisy_waveforms_spectrograms.png",dpi=160)


target= 0 dB | measured=-0.000 dB | alpha=0.547851 | clean RMS=0.0917
target= 5 dB | measured= 5.000 dB | alpha=0.308080 | clean RMS=0.0917
target=10 dB | measured=10.000 dB | alpha=0.173246 | clean RMS=0.0917


## Task 2 — STFT, spectral subtraction và Wiener filtering

In [31]:
def enhance(y, noise_clip, beta=1.5, floor=0.02):
    Y=stft(y)
    N=stft(noise_clip)
    Pn=np.mean(np.abs(N)**2,axis=1,keepdims=True)
    Py=np.abs(Y)**2
    Pss=np.maximum(Py-beta*Pn,floor*Pn)
    Sss=np.sqrt(Pss)*np.exp(1j*np.angle(Y))
    ss=istft(Sss,len(y))
    Ps=np.maximum(Py-Pn,0.0); H=Ps/(Ps+Pn+1e-12)
    w=istft(H*Y,len(y))
    return ss,w

# Thí nghiệm tham số trên noisy speech SNR=5 dB.
y5=noisy_by_snr[5]
noise_base=match_length(noise,min(len(noise),SR*3))
noise_ref=alpha_by_snr[5]*noise_base  # PSD calibration: estimate the actual scaled additive noise
param_rows=[]
for beta in BETAS:
  for floor in FLOORS:
    ss,w=enhance(y5,noise_ref,beta=beta,floor=floor)
    param_rows.append({"beta":beta,"floor":floor,"SNRout_SS_dB":measured_snr(clean,ss),"DeltaSNR_SS_dB":measured_snr(clean,ss)-5,"SNRout_Wiener_dB":measured_snr(clean,w),"DeltaSNR_Wiener_dB":measured_snr(clean,w)-5})
param_table=pd.DataFrame(param_rows)
display(param_table.round(3))
# Required reference setting; also run at all input SNRs.
ss5,w5=enhance(y5,noise_ref,beta=1.5,floor=0.02)
save_audio("enhanced_ss.wav",ss5); save_audio("enhanced_wiener.wav",w5)
from IPython.display import Audio, display
display(Audio(y5,rate=SR), Audio(ss5,rate=SR), Audio(w5,rate=SR))
enhanced_by_snr={}
for target,y in noisy_by_snr.items():
  ss,w=enhance(y,alpha_by_snr[target]*noise_base,beta=1.5,floor=0.02); enhanced_by_snr[target]=(ss,w)
  print(f"SNR_in={target:2d} dB | SS out={measured_snr(clean,ss):6.2f} (Δ {measured_snr(clean,ss)-target:+.2f}) | Wiener out={measured_snr(clean,w):6.2f} (Δ {measured_snr(clean,w)-target:+.2f})")
fig,axs=plt.subplots(2,2,figsize=(12,8))
plot_spec(axs[0,0],y5,"Noisy 5 dB"); plot_spec(axs[0,1],ss5,"Spectral subtraction")
plot_spec(axs[1,0],w5,"Wiener filtering"); plot_spec(axs[1,1],clean,"Clean reference")
fig.tight_layout(); fig.savefig(FIG/"enhancement_spectrograms.png",dpi=160)


,beta,floor,SNRout_SS_dB,DeltaSNR_SS_dB,SNRout_Wiener_dB,DeltaSNR_Wiener_dB
0,1.0,0.00,8.052,3.052,8.891,3.891
1,1.0,0.01,7.976,2.976,8.891,3.891
2,1.0,0.05,7.847,2.847,8.891,3.891
3,1.5,0.00,8.798,3.798,8.891,3.891
4,1.5,0.01,8.800,3.800,8.891,3.891
5,1.5,0.05,8.711,3.711,8.891,3.891
6,2.0,0.00,8.930,3.930,8.891,3.891
7,2.0,0.01,9.066,4.066,8.891,3.891
8,2.0,0.05,9.072,4.072,8.891,3.891


SNR_in= 0 dB | SS out=  5.78 (Δ +5.78) | Wiener out=  5.74 (Δ +5.74)
SNR_in= 5 dB | SS out=  8.78 (Δ +3.78) | Wiener out=  8.89 (Δ +3.89)
SNR_in=10 dB | SS out= 12.31 (Δ +2.31) | Wiener out= 12.36 (Δ +2.36)


## Task 3 — MFCC, enrollment, speaker identification và ablation

In [32]:
def embedding(x, mode="mfcc_delta", cmn=False):
    M=mfcc_features(x)
    if cmn: M=M-M.mean(axis=1,keepdims=True)
    if mode=="mfcc": F=M
    else: F=np.vstack([M,delta_features(M)])
    e=np.r_[F.mean(axis=1),F.std(axis=1)]
    return e/(np.linalg.norm(e)+1e-12)
def cosine(a,b): return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))
def build_models(mode="mfcc_delta",cmn=False,n_enroll=4):
    models={}
    for spk in SPEAKERS:
        files=sorted((DATA/"clean").glob(f"{spk}_enroll_*.wav"))[:n_enroll]
        if not files: raise ValueError(f"No enrollment files for {spk}")
        E=np.vstack([embedding(load_audio(f),mode,cmn) for f in files])
        m=E.mean(axis=0); models[spk]=m/(np.linalg.norm(m)+1e-12)
    return models
def score_one(x,models,mode="mfcc_delta",cmn=False):
    e=embedding(x,mode,cmn); return {s:cosine(e,m) for s,m in models.items()}
def split_files(spk,split):
    if split=="dev": return [DEV_FILES[spk]]
    return sorted((DATA/"clean").glob(f"{spk}_{split}_*.wav"))
def eval_split(split,models,mode="mfcc_delta",cmn=False):
    rows=[]
    for spk in SPEAKERS:
      for f in split_files(spk,split):
        scores=score_one(load_audio(f),models,mode,cmn)
        rows.append({"file":f.name,"true":spk,"pred":max(scores,key=scores.get),**scores})
    return pd.DataFrame(rows)
models=build_models()
test_id=eval_split("test",models)
print("Test speaker-ID accuracy:",np.mean(test_id.true==test_id.pred))
display(test_id.round(3))
cm=confusion_matrix(test_id.true,test_id.pred,labels=SPEAKERS)
fig,ax=plt.subplots(figsize=(5,4)); ConfusionMatrixDisplay(cm,display_labels=SPEAKERS).plot(ax=ax,cmap="Blues",colorbar=False,values_format="d")
ax.set_title("Speaker identification — test"); fig.tight_layout(); fig.savefig(FIG/"confusion_matrix.png",dpi=160)
# Ablations: MFCC-only vs MFCC+delta; CMN off vs on; enrollment count.
abl=[]
for mode in ["mfcc","mfcc_delta"]:
 for cmn in [False,True]:
  for n in [1,2,4]:
   m=build_models(mode,cmn,n); ev=eval_split("test",m,mode,cmn)
   abl.append({"features":mode,"CMN":cmn,"enrollment_per_speaker":n,"accuracy":np.mean(ev.true==ev.pred)})
abl_table=pd.DataFrame(abl); display(abl_table)


Test speaker-ID accuracy: 1.0


,file,true,pred,spk01,spk02,spk03
0,spk01_test_01.wav,spk01,spk01,0.983,0.960,0.956
1,spk01_test_02.wav,spk01,spk01,0.996,0.977,0.929
2,spk02_test_01.wav,spk02,spk02,0.965,0.988,0.945
3,spk02_test_02.wav,spk02,spk02,0.975,0.996,0.919
4,spk03_test_01.wav,spk03,spk03,0.921,0.906,0.992
5,spk03_test_02.wav,spk03,spk03,0.952,0.941,0.997


C:\Users\HP\AppData\Local\Temp\ipykernel_1552\1031916507.py:34: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig,ax=plt.subplots(figsize=(5,4)); ConfusionMatrixDisplay(cm,display_labels=SPEAKERS).plot(ax=ax,cmap="Blues",colorbar=False,values_format="d")


,features,CMN,enrollment_per_speaker,accuracy
0,mfcc,False,1,1.000000
1,mfcc,False,2,1.000000
2,mfcc,False,4,1.000000
3,mfcc,True,1,0.833333
4,mfcc,True,2,0.833333
5,mfcc,True,4,0.666667
6,mfcc_delta,False,1,1.000000
7,mfcc_delta,False,2,1.000000
8,mfcc_delta,False,4,1.000000
9,mfcc_delta,True,1,0.833333


## Task 4 — Speaker verification, chọn threshold trên dev và test độc lập

In [33]:
def trials(split,models):
    genuine=[]; impostor=[]
    for true_spk in SPEAKERS:
      for f in split_files(true_spk,split):
        x=load_audio(f)
        for claimed in SPEAKERS:
          z=score_one(x,models)[claimed]
          (genuine if claimed==true_spk else impostor).append(z)
    return np.asarray(genuine),np.asarray(impostor)
def far_frr(genuine,impostor,theta):
    return np.mean(impostor>=theta),np.mean(genuine<theta)
models_verification=build_models(n_enroll=max(1,min(3,min(len(list(CLEAN_DIR.glob(f"{spk}_enroll_*.wav"))) for spk in SPEAKERS))))
g_dev,i_dev=trials("dev",models_verification)
thresholds=np.linspace(-1,1,401)
curve=np.array([[th,*far_frr(g_dev,i_dev,th)] for th in thresholds])
idx=np.argmin(np.abs(curve[:,1]-curve[:,2])); THETA=float(curve[idx,0])
# Threshold is frozen here; only then evaluate independent test trials.
g_test,i_test=trials("test",models_verification)
FAR_TEST,FRR_TEST=far_frr(g_test,i_test,THETA)
print(f"Frozen dev threshold={THETA:.3f}; dev FAR={curve[idx,1]:.3f}, dev FRR={curve[idx,2]:.3f}")
print(f"Independent test FAR={FAR_TEST:.3f}, FRR={FRR_TEST:.3f}; genuine n={len(g_test)}, impostor n={len(i_test)}")
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(curve[:,0],curve[:,1],label="FAR dev"); ax.plot(curve[:,0],curve[:,2],label="FRR dev"); ax.axvline(THETA,color="k",ls="--",label=f"locked θ={THETA:.2f}")
ax.set(xlabel="Threshold θ",ylabel="Rate",ylim=(-.03,1.03),title="Verification threshold selection on dev"); ax.grid(alpha=.25); ax.legend(); fig.tight_layout(); fig.savefig(FIG/"far_frr_dev.png",dpi=160)


Frozen dev threshold=0.970; dev FAR=0.000, dev FRR=0.000
Independent test FAR=0.167, FRR=0.000; genuine n=6, impostor n=12


## Task 5 — Kiểm tra speaker-ID trên clean, noisy và enhanced speech

In [34]:
# Cùng một utterance CASE_SPEAKER_test_01; dự đoán trên các điều kiện tín hiệu.
integration=[]
for snr_db, (ss,w) in enhanced_by_snr.items():
    for label,x in [(f"noisy_{snr_db}dB",noisy_by_snr[snr_db]),(f"SS_{snr_db}dB",ss),(f"Wiener_{snr_db}dB",w)]:
        scores=score_one(x,models); integration.append({"condition":label,"predicted":max(scores,key=scores.get),"correct":max(scores,key=scores.get)==CASE_SPEAKER,"true_score":scores[CASE_SPEAKER],"margin":scores[CASE_SPEAKER]-max(v for k,v in scores.items() if k!=CASE_SPEAKER)})
xclean_scores=score_one(clean,models)
integration.insert(0,{"condition":"clean","predicted":max(xclean_scores,key=xclean_scores.get),"correct":max(xclean_scores,key=xclean_scores.get)==CASE_SPEAKER,"true_score":xclean_scores[CASE_SPEAKER],"margin":xclean_scores[CASE_SPEAKER]-max(v for k,v in xclean_scores.items() if k!=CASE_SPEAKER)})
integration_table=pd.DataFrame(integration); display(integration_table.round(3))


,condition,predicted,correct,true_score,margin
0,clean,spk01,True,0.983,0.023
1,noisy_0dB,spk03,False,0.752,-0.042
2,SS_0dB,spk03,False,0.910,-0.025
3,Wiener_0dB,spk03,False,0.922,-0.018
4,noisy_5dB,spk03,False,0.845,-0.035
5,SS_5dB,spk03,False,0.942,-0.013
6,Wiener_5dB,spk03,False,0.949,-0.007
7,noisy_10dB,spk03,False,0.902,-0.025
8,SS_10dB,spk03,False,0.960,-0.001
9,Wiener_10dB,spk01,True,0.964,0.004


## Task 6 — Hai microphone mixture, FastICA và đánh giá permutation

In [35]:
def rms_normalize(x): return x/(rms(x)+1e-12)
def corr_abs_matrix(S,Sh):
    return np.abs(np.corrcoef(np.vstack([S,Sh]))[:2,2:])
def best_match(S,Sh):
    C=corr_abs_matrix(S,Sh); rows,cols=linear_sum_assignment(-C)
    order=np.empty(2,dtype=int); order[rows]=cols
    aligned=Sh[order].copy()
    signs=np.sign([np.corrcoef(S[i],aligned[i])[0,1] for i in range(2)])
    aligned*=signs[:,None]
    return C,aligned,order,signs
Sfiles=[DATA/"separation"/"source_A.wav",DATA/"separation"/"source_B.wav"]
sources=[load_audio(f) for f in Sfiles]; L=min(map(len,sources)); S=np.vstack([rms_normalize(x[:L]) for x in sources])
matrix_path=DATA/"metadata"/"separation_mixing_matrices.json"
if matrix_path.exists():
    matrices=json.loads(matrix_path.read_text(encoding="utf-8"))
else:
    matrices={"A_primary":{"matrix":[[1.0,0.6],[0.4,1.0]]},"A_alternative":{"matrix":[[1.0,0.85],[0.8,1.0]]}}
    print("No mixing-matrix metadata found; using the two matrices specified by Lab 5.")
separation_results={}
for key in ["A_primary","A_alternative"]:
    A=np.asarray(matrices[key]["matrix"],dtype=float); X=A@S
    X_wav=0.99*X/(np.max(np.abs(X))+1e-12)  # common gain preserves the two-channel mixture geometry
    print(f"{key}: det(A)={np.linalg.det(A):.4f}")
    save_audio(f"mix_{key}_mic1.wav",X_wav[0]); save_audio(f"mix_{key}_mic2.wav",X_wav[1])
    if key=="A_primary":
        save_audio("mix_mic1.wav",X_wav[0]); save_audio("mix_mic2.wav",X_wav[1])
    # Do not provide A to ICA; only observed mixtures are given.
    ica=FastICA(n_components=2,whiten="unit-variance",random_state=RNG_SEED,max_iter=2000,tol=1e-4)
    Sh=ica.fit_transform(X.T).T
    C,aligned,order,signs=best_match(S,Sh)
    C_after=corr_abs_matrix(S,aligned)
    print("|rho| before matching:\n",np.round(C,3)); print("|rho| after matching/sign alignment:\n",np.round(C_after,3))
    if key=="A_primary":
      for i in range(2): save_audio(f"separated_{i+1}.wav",aligned[i],peak_normalize=True)
      separation_results[key]=(X,aligned,C,C_after)
    else: separation_results[key]=(X,aligned,C,C_after)
# Reference output spectrograms and correlation figure.
X,Sh,C,Ca=separation_results["A_primary"]
fig,axs=plt.subplots(2,2,figsize=(11,7)); plot_spec(axs[0,0],X[0],"Mixture microphone 1"); plot_spec(axs[0,1],X[1],"Mixture microphone 2"); plot_spec(axs[1,0],Sh[0],"Separated source 1"); plot_spec(axs[1,1],Sh[1],"Separated source 2")
fig.tight_layout(); fig.savefig(FIG/"ica_waveforms_spectrograms.png",dpi=160)
fig,axs=plt.subplots(1,2,figsize=(8,3.5))
for ax,M,title in [(axs[0],C,"Before matching"),(axs[1],Ca,"After matching")]:
 im=ax.imshow(M,vmin=0,vmax=1,cmap="viridis"); ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["estimate 1","estimate 2"],yticklabels=["source A","source B"],title=title)
 for (i,j),v in np.ndenumerate(M): ax.text(j,i,f"{v:.2f}",ha="center",va="center",color="white")
fig.colorbar(im,ax=axs.ravel().tolist(),label="|correlation|"); fig.savefig(FIG/"ica_correlation_matrix.png",dpi=160,bbox_inches="tight")


No mixing-matrix metadata found; using the two matrices specified by Lab 5.
A_primary: det(A)=0.7600
|rho| before matching:
 [[1.    0.002]
 [0.014 1.   ]]
|rho| after matching/sign alignment:
 [[1.    0.002]
 [0.014 1.   ]]
A_alternative: det(A)=0.3200
|rho| before matching:
 [[1.    0.001]
 [0.014 1.   ]]
|rho| after matching/sign alignment:
 [[1.    0.001]
 [0.014 1.   ]]


## Task 7 — ICA khi mixture có nhiễu nhẹ

In [36]:
rng=np.random.default_rng(RNG_SEED)
A=np.asarray(matrices["A_primary"]["matrix"],float); X=A@S
noise_level=0.03*np.std(X)
Xn=X+rng.normal(0,noise_level,size=X.shape)
def run_ica_eval(observed):
    est=FastICA(n_components=2,whiten="unit-variance",random_state=RNG_SEED,max_iter=2000,tol=1e-4).fit_transform(observed.T).T
    C,aligned,_,_=best_match(S,est)
    return C,corr_abs_matrix(S,aligned)
C_clean,Cclean_after=run_ica_eval(X); C_noisy,Cnoisy_after=run_ica_eval(Xn)
print("clean |rho| matched:",np.round(Cclean_after,3)); print("noisy |rho| matched:",np.round(Cnoisy_after,3))
print("mean matched diagonal |rho| clean/noisy:",np.diag(Cclean_after).mean(),np.diag(Cnoisy_after).mean())


clean |rho| matched: [[1.    0.002]
 [0.014 1.   ]]
noisy |rho| matched: [[0.999 0.001]
 [0.016 0.999]]
mean matched diagonal |rho| clean/noisy: 0.9999480471178838 0.9986962280002272


## Task 8 — Tóm tắt số liệu, tái lập và lưu bảng

In [37]:
snr_rows=[]
for target,y in noisy_by_snr.items():
  ss,w=enhanced_by_snr[target]
  snr_rows.append({"SNR_in_dB":target,"SNR_measured_dB":measured_snr(clean,y),"SS_SNR_out_dB":measured_snr(clean,ss),"SS_delta_dB":measured_snr(clean,ss)-target,"Wiener_SNR_out_dB":measured_snr(clean,w),"Wiener_delta_dB":measured_snr(clean,w)-target})
snr_table=pd.DataFrame(snr_rows); display(snr_table.round(3))
test_id.to_csv(OUT/"speaker_identification_test.csv",index=False,encoding="utf-8-sig")
abl_table.to_csv(OUT/"speaker_ablation.csv",index=False,encoding="utf-8-sig")
param_table.to_csv(OUT/"enhancement_parameter_sweep.csv",index=False,encoding="utf-8-sig")
snr_table.to_csv(OUT/"enhancement_snr.csv",index=False,encoding="utf-8-sig")
print("Saved audio and figures under",OUT)


,SNR_in_dB,SNR_measured_dB,SS_SNR_out_dB,SS_delta_dB,Wiener_SNR_out_dB,Wiener_delta_dB
0,0,-0.0,5.785,5.785,5.742,5.742
1,5,5.0,8.782,3.782,8.891,3.891
2,10,10.0,12.308,2.308,12.365,2.365


Saved audio and figures under d:\CSE457-Audio-Speech-Processing\Lab05_2351260681_NguyenMinhPhuc\results
